# Import Libraries

In [1]:
import pandas as pd
import os
import glob

# 1. Define input and output folders

In [2]:
RAW_FOLDER = "../Fifa worldcup data/row"
CLEANED_FOLDER = "../Fifa worldcup data/cleaned"

# Create cleaned folder if it does not exist
os.makedirs(CLEANED_FOLDER, exist_ok=True)

print("Current working directory:")
print(os.getcwd())

print("\nRaw folder:")
print(os.path.abspath(RAW_FOLDER))

print("\nCleaned folder:")
print(os.path.abspath(CLEANED_FOLDER))

Current working directory:
c:\Users\shima\Desktop\FIFA-World-Cup-Data-Analytics\Python

Raw folder:
c:\Users\shima\Desktop\FIFA-World-Cup-Data-Analytics\Fifa worldcup data\row

Cleaned folder:
c:\Users\shima\Desktop\FIFA-World-Cup-Data-Analytics\Fifa worldcup data\cleaned


# 2. FIND ALL CSV FILES

In [3]:
print("\n" + "=" * 70)
print("FIFA WORLD CUP DATA CLEANING")
print("=" * 70)

# Check if raw folder exists
csv_files = []

if not os.path.exists(RAW_FOLDER):

    print("\n❌ ERROR: Raw folder does not exist!")
    print("Expected location:")
    print(os.path.abspath(RAW_FOLDER))

else:

    print("\n✅ Raw folder exists.")

    # Find CSV files
    csv_files = glob.glob(
        os.path.join(RAW_FOLDER, "*.csv")
    )

    print(
        f"\nNumber of CSV files found: {len(csv_files)}"
    )

    # Print file names
    for file in sorted(csv_files):
        print(" -", os.path.basename(file))


FIFA WORLD CUP DATA CLEANING

✅ Raw folder exists.

Number of CSV files found: 17
 - Bookings.csv
 - Confederations.csv
 - FIFA_Rankings.csv
 - Federations.csv
 - Goals.csv
 - Match_Attendance.csv
 - Matches.csv
 - Passing_Network_Edges.csv
 - Player_Appearances.csv
 - Player_Match_Stats.csv
 - Players.csv
 - Stadiums.csv
 - Substitutions.csv
 - Team_Appearances.csv
 - Team_Match_Stats.csv
 - Teams.csv
 - Tournaments.csv


# 3. Process each CSV file

In [4]:
if len(csv_files) == 0:

    print("\n❌ No CSV files found.")
    print("Check that your CSV files are inside:")
    print(os.path.abspath(RAW_FOLDER))

else:

    print("\n" + "=" * 70)
    print("STARTING CLEANING")
    print("=" * 70)

    cleaning_summary = []

    for file in sorted(csv_files):

        file_name = os.path.basename(file)

        print("\n" + "=" * 70)
        print(f"PROCESSING: {file_name}")
        print("=" * 70)

        # ----------------------------------------------------
        # READ CSV
        # ----------------------------------------------------

        try:

            df = pd.read_csv(file)

        except Exception as e:

            print("❌ Error reading file:")
            print(e)
            continue

        original_rows = len(df)
        original_columns = len(df.columns)

        print(f"Original rows    : {original_rows}")
        print(f"Original columns : {original_columns}")

        # ----------------------------------------------------
        # REMOVE COMPLETELY EMPTY ROWS
        # ----------------------------------------------------

        empty_rows = df.isna().all(axis=1).sum()

        df = df.dropna(how="all")

        # ----------------------------------------------------
        # REMOVE DUPLICATE ROWS
        # ----------------------------------------------------

        duplicate_count = df.duplicated().sum()

        if duplicate_count > 0:

            df = df.drop_duplicates()

        # ----------------------------------------------------
        # CLEAN COLUMN NAMES
        # ----------------------------------------------------

        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        # ----------------------------------------------------
        # CLEAN TEXT COLUMNS
        # ----------------------------------------------------

        text_columns = df.select_dtypes(
            include="object"
        ).columns

        for column in text_columns:

            df[column] = df[column].apply(
                lambda x:
                    x.strip()
                    if isinstance(x, str)
                    else x
            )

        # ----------------------------------------------------
        # CONVERT DATE COLUMNS
        # ----------------------------------------------------

        date_columns = [
            "birth_date",
            "start_date",
            "end_date",
            "match_date"
        ]

        for column in date_columns:

            if column in df.columns:

                nulls_before = df[column].isna().sum()

                df[column] = pd.to_datetime(
                    df[column],
                    errors="coerce"
                )

                new_nat = df[column].isna().sum() - nulls_before

                if new_nat > 0:
                    print(f"⚠️ {column}: {new_nat} values could not be parsed -> NaT")

        # ----------------------------------------------------
        # SAVE CLEANED FILE
        # ----------------------------------------------------

        output_file = os.path.join(
            CLEANED_FOLDER,
            file_name
        )

        df.to_csv(
            output_file,
            index=False
        )

        # ----------------------------------------------------
        # PRINT SUMMARY
        # ----------------------------------------------------

        final_rows = len(df)

        print(f"Empty rows removed : {empty_rows}")
        print(f"Duplicates removed : {duplicate_count}")
        print(f"Final rows         : {final_rows}")
        print(f"Saved to           : {output_file}")

        # Add information to summary
        cleaning_summary.append({

            "file": file_name,
            "original_rows": original_rows,
            "final_rows": final_rows,
            "columns": len(df.columns),
            "empty_rows_removed": empty_rows,
            "duplicates_removed": duplicate_count

        })


STARTING CLEANING

PROCESSING: Bookings.csv
Original rows    : 3459
Original columns : 12
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 3459
Saved to           : ../Fifa worldcup data/cleaned\Bookings.csv

PROCESSING: Confederations.csv
Original rows    : 6
Original columns : 4
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 6
Saved to           : ../Fifa worldcup data/cleaned\Confederations.csv

PROCESSING: FIFA_Rankings.csv
Original rows    : 71019
Original columns : 6


C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(
C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.ht

Empty rows removed : 0
Duplicates removed : 0
Final rows         : 71019
Saved to           : ../Fifa worldcup data/cleaned\FIFA_Rankings.csv

PROCESSING: Federations.csv
Original rows    : 91
Original columns : 4
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 91
Saved to           : ../Fifa worldcup data/cleaned\Federations.csv

PROCESSING: Goals.csv
Original rows    : 3945
Original columns : 11
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 3945
Saved to           : ../Fifa worldcup data/cleaned\Goals.csv

PROCESSING: Match_Attendance.csv
Original rows    : 964
Original columns : 5
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 964
Saved to           : ../Fifa worldcup data/cleaned\Match_Attendance.csv

PROCESSING: Matches.csv
Original rows    : 1352
Original columns : 22
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 1352
Saved to           : ../Fifa worldcup data/cleaned\Matches.csv

PROCESSING: Passing

C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(
C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.ht

Empty rows removed : 0
Duplicates removed : 0
Final rows         : 52072
Saved to           : ../Fifa worldcup data/cleaned\Passing_Network_Edges.csv

PROCESSING: Player_Appearances.csv
Original rows    : 32824
Original columns : 20


...

C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(
C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.ht

Empty rows removed : 0
Duplicates removed : 0
Final rows         : 11364
Saved to           : ../Fifa worldcup data/cleaned\Players.csv

PROCESSING: Stadiums.csv
Original rows    : 256
Original columns : 5
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 256
Saved to           : ../Fifa worldcup data/cleaned\Stadiums.csv

PROCESSING: Substitutions.csv
Original rows    : 12220
Original columns : 10
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 12220
Saved to           : ../Fifa worldcup data/cleaned\Substitutions.csv

PROCESSING: Team_Appearances.csv
Original rows    : 2704
Original columns : 17
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 2704
Saved to           : ../Fifa worldcup data/cleaned\Team_Appearances.csv

PROCESSING: Team_Match_Stats.csv
Original rows    : 16120
Original columns : 14


C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(
C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.ht

Empty rows removed : 0
Duplicates removed : 0
Final rows         : 16120
Saved to           : ../Fifa worldcup data/cleaned\Team_Match_Stats.csv

PROCESSING: Teams.csv
Original rows    : 93
Original columns : 7
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 93
Saved to           : ../Fifa worldcup data/cleaned\Teams.csv

PROCESSING: Tournaments.csv
Original rows    : 31
Original columns : 18
Empty rows removed : 0
Duplicates removed : 0
Final rows         : 31
Saved to           : ../Fifa worldcup data/cleaned\Tournaments.csv


C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(
C:\Users\shima\AppData\Local\Temp\ipykernel_32272\811535077.py:76: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.ht

# 4. CLEANING SUMMARY

In [5]:
if len(cleaning_summary) > 0:

    summary_df = pd.DataFrame(
        cleaning_summary
    )

    print("\n" + "=" * 70)
    print("CLEANING SUMMARY")
    print("=" * 70)

    print(
        summary_df.to_string(index=False)
    )

    # Save summary
    summary_file = os.path.join(
        CLEANED_FOLDER,
        "cleaning_summary.csv"
    )

    summary_df.to_csv(
        summary_file,
        index=False
    )

    print("\n" + "=" * 70)
    print("SUMMARY SAVED")
    print("=" * 70)

    print(summary_file)

    # Total records
    total_records = summary_df["final_rows"].sum()

    print(
        f"\nTotal records after cleaning: {total_records}"
    )

else:

    print("\n❌ No files were processed.")


CLEANING SUMMARY
                     file  original_rows  final_rows  columns  empty_rows_removed  duplicates_removed
             Bookings.csv           3459        3459       12                   0                   0
       Confederations.csv              6           6        4                   0                   0
        FIFA_Rankings.csv          71019       71019        6                   0                   0
          Federations.csv             91          91        4                   0                   0
                Goals.csv           3945        3945       11                   0                   0
     Match_Attendance.csv            964         964        5                   0                   0
              Matches.csv           1352        1352       22                   0                   0
Passing_Network_Edges.csv          52072       52072        9                   0                   0
   Player_Appearances.csv          32824       32824       20   